In [ ]:
# 🧠 Brain Tumor Detection using ResNet50 + Augmentation + Cross-Validation + Class Check

# ✅ Step 1: Install dependencies (only in Colab)
!pip install kaggle nibabel opencv-python-headless matplotlib seaborn scikit-learn --quiet

# ✅ Step 2: Mount Google Drive (Optional)
from google.colab import drive
drive.mount('/content/drive')

# ✅ Step 3: Download BraTS dataset from Kaggle
from google.colab import files
files.upload()  # Upload kaggle.json
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d awsaf49/brats20-dataset-training-validation
!unzip -q brats20-dataset-training-validation.zip -d BraTS/

# ✅ Step 4: Import required libraries
import os, numpy as np, nibabel as nib, cv2, gc
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix, classification_report
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# ✅ Step 5: Define MRI loading function
def load_mri_image(file_path, target_size=(224, 224)):
    img = nib.load(file_path).get_fdata()
    img = np.mean(img, axis=-1)
    img = cv2.resize(img, target_size)
    img = img / np.max(img) if np.max(img) != 0 else img
    img = np.repeat(img[..., np.newaxis], 3, axis=-1)
    return img.astype(np.float32)

# ✅ Step 6: Load data
DATASET_PATH = "BraTS/MICCAI_BraTS2020_TrainingData"
X, y = [], []
for folder in os.listdir(DATASET_PATH):
    folder_path = os.path.join(DATASET_PATH, folder)
    t1 = os.path.join(folder_path, f"{folder}_t1.nii.gz")
    flair = os.path.join(folder_path, f"{folder}_flair.nii.gz")
    if os.path.exists(t1) and os.path.exists(flair):
        img1 = load_mri_image(t1)
        img2 = load_mri_image(flair)
        X.append((img1 + img2) / 2)
        y.append(1 if "HGG" in folder else 0)
X = np.array(X)
y = np.array(y)

# ✅ Step 7: Data Augmentation
aug = ImageDataGenerator(
    rotation_range=10,
    zoom_range=0.1,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)

# ✅ Step 8: Cross-validation with class-check and training visualization
total_folds = 5
kf = StratifiedKFold(n_splits=total_folds, shuffle=True, random_state=42)
fold = 1

for train_index, test_index in kf.split(X, y):
    print(f"\n📂 Fold {fold}/{total_folds}")
    print("🔎 Test class distribution:", dict(zip(*np.unique(y[test_index], return_counts=True))))

    if len(np.unique(y[test_index])) < 2:
        print("⚠️ Skipping fold due to single-class test set.")
        fold += 1
        continue

    X_train, X_test = X[train_index], X[test_index]
    y_train = tf.keras.utils.to_categorical(y[train_index], 2)
    y_test = tf.keras.utils.to_categorical(y[test_index], 2)

    base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    for layer in base_model.layers[:100]:
        layer.trainable = False
    for layer in base_model.layers[100:]:
        layer.trainable = True

    x = GlobalAveragePooling2D()(base_model.output)
    x = Dense(512, activation='relu')(x)
    x = Dropout(0.5)(x)
    output = Dense(2, activation='softmax')(x)

    model = Model(inputs=base_model.input, outputs=output)
    model.compile(optimizer=Adam(1e-4), loss='categorical_crossentropy', metrics=['accuracy'])

    callbacks = [
        EarlyStopping(patience=5, monitor='val_loss', restore_best_weights=True),
        ModelCheckpoint(f"fold{fold}_best_model.keras", save_best_only=True)
    ]

    history = model.fit(
        aug.flow(X_train, y_train, batch_size=16),
        validation_data=(X_test, y_test),
        epochs=20,
        callbacks=callbacks,
        verbose=1
    )

    # 📊 Visualize training history
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Val Loss')
    plt.title(f'Fold {fold} - Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history.history['accuracy'], label='Train Accuracy')
    plt.plot(history.history['val_accuracy'], label='Val Accuracy')
    plt.title(f'Fold {fold} - Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()

    plt.tight_layout()
    plt.show()

    print(f"📈 Evaluating Fold {fold}...")
    y_pred = np.argmax(model.predict(X_test), axis=1)
    y_true = np.argmax(y_test, axis=1)

    labels = [0, 1]
    names = ["No Tumor", "Tumor"]

    cm = confusion_matrix(y_true, y_pred, labels=labels)
    report = classification_report(y_true, y_pred, labels=labels, target_names=names, zero_division=0)

    print("Confusion Matrix:\n", cm)
    print("Classification Report:\n", report)

    tf.keras.backend.clear_session()
    gc.collect()
    fold += 1
